<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/02_machine_learning/reinforcement_learning/experiment_dqn_reinforcement_learning_basic_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

# Define the ReplayBuffer class
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state):
        self.buffer.append((state, action, reward, next_state))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 32),
            nn.ReLU(),
            nn.Linear(32, action_size)
        )

    def forward(self, x):
        return self.net(x)

class DQNAgent:
    def __init__(self, state_size, action_size):
        self.q_net = DQN(state_size, action_size)
        self.target_net = DQN(state_size, action_size)
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=0.001)

        self.buffer = ReplayBuffer() # Now ReplayBuffer is defined
        self.gamma = 0.9
        self.epsilon = 0.1
        self.action_size = action_size

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_size - 1)
        state = torch.tensor([state], dtype=torch.float32)
        return torch.argmax(self.q_net(state)).item()

    def train(self, batch_size=32):
        if len(self.buffer.buffer) < batch_size:
            return

        batch = self.buffer.sample(batch_size)

        for s, a, r, ns in batch:
            s = torch.tensor([s], dtype=torch.float32)
            ns = torch.tensor([ns], dtype=torch.float32)

            target = r + self.gamma * torch.max(self.target_net(ns)).item()
            pred = self.q_net(s)[a]

            loss = (pred - target) ** 2

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())